# 04. SatChecker Query for DECam Streak
Written by Kiyoaki Okudaira and Meredith Rawls<br>
*University of Washington / IAU CPS SatHub<br>
(kiyoaki@uw.edu or okudaira.kiyoaki.528@s.kyushu-u.ac.jp)<br>
<br>
This code is written for ASTR 499 undergraduate research with Dr. Meredith.<br>
A notebook to query the SatChecker Field Of View (FOV) service.<br>
<br>
**History**<br>
coding 2026-02-21 : 1st coding<br>
update 2026-05-31 : enhance readability<br>
bugfix 2026-05-31 : DECam CCD sections coordinate modified

### Import and initial settings
**Standard libraries**

In [ ]:
from os import path
import pickle
import json
from tqdm.notebook import tqdm

import requests

import astropy.units as u
from astropy.time import Time
from astropy.coordinates import EarthLocation
from astropy.io import fits
from astropy.coordinates import SkyCoord

from concurrent.futures import ThreadPoolExecutor, as_completed

**Import file settings**

In [ ]:
# project directory
PATH_project = '/astro/store/shire/kiyoaki/ASTR499/'

PATH_input   = PATH_project + 'input/'
PATH_output  = PATH_project + 'output/'

PATH_image   = PATH_input  + 'fits_data/'
PATH_session = PATH_output + 'session/'

# ORIGINAL streak list from Alex
fname_streak_list = 'streaks_augmented_20230817.csv'
PATH_streak_list  = fname_streak_list + 'decam_streak_list/' + fname_streak_list

**Process method setting**

In [ ]:
paralell_process = True
if paralell_process:
    from concurrent.futures import ThreadPoolExecutor, as_completed

### Load Streak Dataset
Dataset is grouped by EXPNUM because if the EXPNUM is the same, the request to satchecker is the same．

In [ ]:
with open(PATH_session+path.splitext(path.basename(fname_streak_list))[0]+'_02_masked_by_gauss.pkl', 'rb') as f:
    streak_table = pickle.load(f)
streak_table = streak_table.group_by("expnum")
output = []

### Hard-wired DECam parameters

Original source: https://github.com/DarkEnergySurvey/drawDECam

In [ ]:
CCDSECTIONS = {
    1: [-13400, -11333, -6313, -2208],
    2: [-13393, -11327, -2054, 2043],
    3: [-13403, -11334, 2203, 6301],
    4: [-11131, -9073, -8449, -4344],
    5: [-11128, -9072, -4187, -89],
    6: [-11128, -9071, 66, 4161],
    7: [-11131, -9071, 4315, 8413],
    8: [-8870, -6818, -10582, -6475],
    9: [-8867, -6816, -6319, -2221],
    10: [-8865, -6814, -2066, 2028],
    11: [-8866, -6814, 2183, 6277],
    12: [-8866, -6812, 6434, 10533],
    13: [-6616, -4566, -12717, -8605],
    14: [-6613, -4565, -8451, -4352],
    15: [-6613, -4565, -4195, -100],
    16: [-6609, -4560, 55, 4149],
    17: [-6610, -4561, 4305, 8399],
    18: [-6609, -4558, 8558, 12664],
    19: [-4366, -2319, -12710, -8603],
    20: [-4359, -2313, -8450, -4353],
    21: [-4360, -2314, -4197, -103],
    22: [-4360, -2313, 53, 4147],
    23: [-4357, -2310, 4306, 8400],
    24: [-4355, -2307, 8555, 12659],
    25: [-2115, -68, -14851, -10733],
    26: [-2113, -67, -10577, -6478],
    27: [-2108, -62, -6322, -2228],
    28: [-2109, -63, -2069, 2024],
    29: [-2105, -59, 2178, 6272],
    30: [-2104, -58, 6431, 10529],
    31: [-2104, -57, 10684, 14802],
    32: [134, 2182, -14850, -10734],
    33: [137, 2184, -10577, -6480],
    34: [140, 2186, -6324, -2230],
    35: [141, 2187, -2073, 2020],
    36: [143, 2189, 2173, 6269],
    37: [143, 2189, 6426, 10526],
    38: [148, 2194, 10684, 14804],
    39: [2387, 4436, -12710, -8607],
    40: [2389, 4437, -8452, -4358],
    41: [2389, 4436, -4200, -106],
    42: [2392, 4439, 50, 4145],
    43: [2393, 4439, 4301, 8399],
    44: [2396, 4443, 8554, 12663],
    45: [4638, 6691, -12716, -8612],
    46: [4640, 6690, -8459, -4365],
    47: [4642, 6691, -4204, -111],
    48: [4643, 6692, 42, 4138],
    49: [4643, 6692, 4298, 8398],
    50: [4645, 6695, 8554, 12668],
    51: [6896, 8952, -10591, -6492],
    52: [6894, 8948, -6337, -2242],
    53: [6893, 8946, -2084, 2010],
    54: [6897, 8949, 2164, 6264],
    55: [6897, 8951, 6420, 10529],
    56: [9155, 11217, -8476, -4378],
    57: [9152, 11212, -4225, -129],
    58: [9152, 11211, 28, 4127],
    59: [9152, 11212, 4282, 8389],
    60: [11421, 13493, -6368, -2269],
    61: [11419, 13490, -2084, 2010],
    62: [11418, 13488, 2143, 6249],
}

CCDSECTION_X0 = (CCDSECTIONS[28][1] + CCDSECTIONS[35][0]) / 2.0
CCDSECTION_Y0 = (CCDSECTIONS[35][2] + CCDSECTIONS[28][3]) / 2.0

TRIM_CCDSECTIONS = CCDSECTIONS.copy()
borderpix = 104  # 208/2. as 208 is the space between chips in pixels
for _k, _v in list(TRIM_CCDSECTIONS.items()):
    (_x1, _x2, _y1, _y2) = _v
    _x1 = _x1 + borderpix
    _x2 = _x2 - borderpix
    _y1 = _y1 + borderpix
    _y2 = _y2 - borderpix
    TRIM_CCDSECTIONS[_k] = [_x1, _x2, _y1, _y2]

def createDECam_TANheader(ra_center, dec_center, pixscale=0.2634):
    """
    Creates a fake TAN projection header for DECam image to project the
    CCD Sections on the sky
    pixscale is in arcseconds per pixel
    """
    DECam_header = {
        'CTYPE1': 'RA---TAN',  # / WCS projection type for this axis
        'CTYPE2': 'DEC--TAN',  # / WCS projection type for this axis
        'CUNIT1': 'deg',  # / Axis unit
        'CUNIT2': 'deg',  # / Axis unit
        'CRVAL1': ra_center,  # / World coordinate on this axis
        'CRPIX1': CCDSECTION_X0,  # / Reference pixel on this axis
        'CD1_1': 0,  # /
        'CD1_2': +pixscale / 3600.,  # /
        'CRVAL2': dec_center,  # /
        'CRPIX2': CCDSECTION_Y0,  # /
        'CD2_1': -pixscale / 3600.,  # /
        'CD2_2': 0.  # /
    }
    return DECam_header

### SatChecker query parameters

Must specify the observatory location, the sky region, and the time and duration of the observation.<br>
Docs available at https://satchecker.readthedocs.io/en/latest/fov.html<br>
<br>
**FOV and Telescope location**

In [ ]:
fov_radius = 1.5  # degree radius for the satchecker query
# DECam FOV is 3 square degrees (2.2 degrees across)
location = EarthLocation.of_site('ctio')
latitude = location.lat.value  # deg
longitude = location.lon.value  # deg
elevation = location.height.value  # meters

**Satchecker query (single processing)**

In [ ]:
if paralell_process is False:
    for group in tqdm(streak_table.groups):
        basename = path.basename(group[0]["archive_filename"])
        md5sum = group[0]["md5sum"]
        expnum = group[0]["expnum"]
        ccdnum = group[0]["ccdnum"]
        streak_ID = group[0]["streakID"]

        save_path = PATH_image + path.splitext(path.splitext(basename)[0])[0] + "_CCD_{0}".format(ccdnum) + path.splitext(path.splitext(basename)[0])[1] + path.splitext(basename)[1]
        satchecker_result_path = PATH_output + "satchecker/" + path.splitext(path.splitext(basename)[0])[0] + "_satchecker.json"

        if path.exists(satchecker_result_path):
            continue

        main_header = fits.open(save_path)[0].header

        exp_time = main_header["EXPTIME"] * u.s
        exp_begin = Time(main_header["DATE-OBS"])

        start_time_jd = exp_begin.jd
        duration = exp_time.value

        # Observation time margin for SatChecker query
        start_time_jd = (exp_begin - exp_time).jd
        duration = (exp_time * 3).value

        # telescope coordinate
        coord = SkyCoord(ra=main_header["TELRA"], dec=main_header["TELDEC"], unit=(u.hour, u.deg))
        ra_center = coord.icrs.ra.value
        dec_center = coord.icrs.dec.value

        # Make the SatChecker API request
        url_string = f"https://satchecker.cps.iau.org/fov/satellite-passes/?latitude={latitude}&longitude={longitude}&elevation={elevation}&start_time_jd={start_time_jd}&duration={duration}&ra={ra_center}&dec={dec_center}&fov_radius={fov_radius}&group_by=satellite&async=False"
        response = requests.get(url_string, timeout=60)
        data = response.json()
        print(f"URL : {url_string}")

        with open(PATH_output + "satchecker/" + path.splitext(path.splitext(basename)[0])[0] + "_satchecker.json" ,'w') as json_output:
            json.dump(data, json_output, ensure_ascii=False, indent=4, sort_keys=True, separators=(',', ': '))

**Satchecker query (paralell processing)**

In [ ]:
if paralell_process:
    def process_one_group(group):

        basename = path.basename(group[0]["archive_filename"])
        md5sum = group[0]["md5sum"]
        expnum = group[0]["expnum"]
        ccdnum = group[0]["ccdnum"]
        streak_ID = group[0]["streakID"]

        save_path = (
            PATH_image
            + path.splitext(path.splitext(basename)[0])[0]
            + f"_CCD_{ccdnum}"
            + path.splitext(path.splitext(basename)[0])[1]
            + path.splitext(basename)[1]
        )

        satchecker_result_path = (
            PATH_output
            + "satchecker/"
            + path.splitext(path.splitext(basename)[0])[0]
            + "_satchecker.json"
        )

        if path.exists(satchecker_result_path):
            return "exist"

        try:
            with fits.open(save_path) as hdul:
                main_header = hdul[0].header

            exp_time = main_header["EXPTIME"] * u.s
            exp_begin = Time(main_header["DATE-OBS"])

            # SatChecker query margin
            start_time_jd = (exp_begin - exp_time).jd
            duration = (exp_time * 3).value

            coord = SkyCoord(
                ra=main_header["TELRA"],
                dec=main_header["TELDEC"],
                unit=(u.hour, u.deg),
            )

            ra_center = coord.icrs.ra.value
            dec_center = coord.icrs.dec.value

            url_string = (
                f"https://satchecker.cps.iau.org/fov/satellite-passes/"
                f"?latitude={latitude}"
                f"&longitude={longitude}"
                f"&elevation={elevation}"
                f"&start_time_jd={start_time_jd}"
                f"&duration={duration}"
                f"&ra={ra_center}"
                f"&dec={dec_center}"
                f"&fov_radius={fov_radius}"
                f"&group_by=satellite"
                f"&async=False"
            )

            response = requests.get(url_string, timeout=60)
            data = response.json()

            with open(satchecker_result_path, "w") as json_output:
                json.dump(
                    data,
                    json_output,
                    ensure_ascii=False,
                    indent=4,
                    sort_keys=True,
                    separators=(",", ": "),
                )

            return "done"

        except Exception as e:
            print(f"error: {e}")
            return f"error: {e}"

    max_workers = 3

    results = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(process_one_group, group)
                for group in streak_table.groups]

        for fut in tqdm(as_completed(futures), total=len(futures)):
            results.append(fut.result())
